In [2]:
from glob import glob
from cmcmultimodal.io import mat2nii
from pathlib import Path
import os

my_files = glob('../tests/testdata/Slice*.mat')
highres_files = []
lowres_files = []
for f in my_files:
    nii_highres, nii_lowres = mat2nii(f, nii_lowres_file=os.path.join(Path(f).parent,'lowres/',
                                      Path(f).name.replace('.mat','.nii.gz')), downsample=10)
    highres_files.append(nii_highres)
    lowres_files.append(nii_lowres)
    
highres_files

[PosixPath('../tests/testdata/Slice_007_EnR.nii.gz'),
 PosixPath('../tests/testdata/Slice_006_EnR.nii.gz'),
 PosixPath('../tests/testdata/Slice_004_EnR.nii.gz'),
 PosixPath('../tests/testdata/Slice_014_EnR.nii.gz'),
 PosixPath('../tests/testdata/Slice_005_EnR.nii.gz'),
 PosixPath('../tests/testdata/Slice_010_EnR.nii.gz'),
 PosixPath('../tests/testdata/Slice_009_EnR.nii.gz'),
 PosixPath('../tests/testdata/Slice_003_EnR.nii.gz'),
 PosixPath('../tests/testdata/Slice_013_EnR.nii.gz'),
 PosixPath('../tests/testdata/Slice_012_EnR.nii.gz')]

In [1]:
from cmcmultimodal.proc import psoct
from pathlib import Path

datadir = '/Users/Vasilis/Downloads/PSOCT_Ret'

my_data = psoct(Path(datadir), lowres=True, slide_range=(98,200), reg_modality='Retardance')
slides, rel_shifts, abs_shifts = my_data.run_registration(bad_slides=[140,])


In [2]:
output_path = '/Users/Vasilis/Downloads/PSOCT_test_output_6'
mri_ref = 'MRI/reoriented_FA.nii.gz'

indiv_slides = my_data.run_slide_deck_creation(slides, abs_shifts, other_images=['Retardance','Orientation'], 
                                               output_path=output_path, mri_ref=mri_ref, downsample=1)

In [ ]:
my_data.apply_to_highres_images(indiv_slides, abs_shifts, ['Retardance','Orientation'])

In [ ]:
# Process all MOE slides

from cmcmultimodal.proc import psoct
from pathlib import Path

datadir = '/Users/Vasilis/Downloads/PSOCT_Ret'
output_path = '/Users/Vasilis/Downloads/PSOCT_all_slides'
mri_ref = '/Users/Vasilis/Downloads/PSOCT_Ret/MRI/reoriented_FA.nii.gz'

my_data = psoct(Path(datadir), lowres=True, reg_modality='Retardance')
slides, rel_shifts, abs_shifts = my_data.run_registration(bad_slides=[140,], verbose=True)


indiv_slides = my_data.run_slide_deck_creation(slides, abs_shifts, other_images=['Retardance'], 
                                               output_path=output_path, mri_ref=mri_ref, downsample=1)

In [ ]:
from fsl.data.image         import Image
import numpy as np
from pathlib import Path

ref_path = Path('/Users/Vasilis/Downloads/PSOCT_test_output_2')
est_path = Path('/Users/Vasilis/Downloads/PSOCT_test_output_3')

def compare_images(ref, est):
    ref_img = Image(ref)
    est_img = Image(est)

    if not np.allclose(ref_img.data, est_img.data):
        print('\t', 'Images are NOT equal!')
    if ref_img.header != est_img.header:
        print('\t', 'Headers are NOT equal!')

def compare_matrices(ref, est):
    ref_mat = np.loadtxt(ref)
    est_mat = np.loadtxt(est)

    if not np.allclose(ref_mat, est_mat, atol=0.001):
        print('\t', 'Matrices are NOT equal!')

for file in ref_path.glob('*'):
    if file.is_dir():
        for subfile in file.glob('*'):
            print(subfile)
            corresponding_est_file = est_path / file.name / subfile.name
            if corresponding_est_file.exists() == False:
                print('\t', 'File does not exist in estimated path!')
                continue
            if subfile.suffix in ['.nii', '.gz']:
                compare_images(subfile, corresponding_est_file)
            elif subfile.suffix == '.mat':
                compare_matrices(subfile, corresponding_est_file)
    else:
        print(file)
        corresponding_est_file = est_path / file.name
        if corresponding_est_file.exists() == False:
            print('\t', 'File does not exist in estimated path!')
            continue
        if file.suffix in ['.nii', '.gz']:
            compare_images(file, corresponding_est_file)
        elif file.suffix == '.mat':
            compare_matrices(file, corresponding_est_file)


In [10]:
# test fslsplit in the three orientations
from fsl.wrappers.avwutils  import fslsplit

inp_path = Path('/Users/Vasilis/Downloads/PSOCT_test_output_3')
out_path = Path('/Users/Vasilis/Downloads/fslsplit_test')
slide_deck = inp_path / 'slide_deck.nii.gz'
orientation = 'sagittal'  

# Lookup table for orientation information
OrientationLookup = {'sagittal': 'x', 'coronal': 'y', 'axial': 'z',
                     'Sagittal': 'x', 'Coronal': 'y', 'Axial': 'z'}

indiv_slides = fslsplit(src=slide_deck, out=out_path/orientation, dim=OrientationLookup[orientation])
